# LSTM-Only Baseline — IBM & Sparkov
Standalone notebook. Runs **only** the pure LSTM baseline (`lstm_solo/`) —
zero graph component, no `edge_index`, no GAT layer anywhere. This is
kept entirely separate from the hybrid GAT+LSTM architectures (which live
in `ibm/`/`sparkov/` and are run by `run_all_colab.ipynb` instead).

Purpose: isolate what the LSTM branch alone achieves, with no relational
signal, for direct comparison against the GATv2-only baseline reported in
the hybrid notebook — this answers whether the graph contributes anything
on top of temporal modelling alone.

| File | Node granularity | Graph strategies |
|---|---|---|
| `lstm_solo/ibm/lstm_only_model.py` | Transaction | none — runs once, not per-strategy |
| `lstm_solo/sparkov/lstm_only_model.py` | Transaction | none — runs once, not per-strategy |

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Update `REPO_URL` in Cell 2
3. Run the IBM section, the Sparkov section, or both — independent, no shared state

---
## Cell 1 — Install dependencies

In [2]:
import subprocess, sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'matplotlib', 'pyarrow'], check=True)
print('Done.')

PyTorch 2.11.0  |  CUDA: cu128
Done.


---
## Cell 2 — Clone the repo
This clones the whole repo (LSTM-only lives inside it at `lstm_solo/`),
but this notebook never touches `ibm/` or `sparkov/`.

In [12]:
import os

REPO_URL = 'https://github.com/Roya62/hybrid-gnn-lstm-fraud.git'

if not os.path.isdir('/content/hybrid-gnn-lstm-fraud'):
    !git clone -q {REPO_URL} /content/hybrid-gnn-lstm-fraud

os.makedirs('/content/outcomes', exist_ok=True)
print('Repo ready at /content/hybrid-gnn-lstm-fraud')

Repo ready at /content/hybrid-gnn-lstm-fraud


---
## Cell 2b — Mount Google Drive
Required: results are saved here incrementally so a runtime disconnect
mid-run doesn't lose completed work.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
# Part A — IBM

## Cell 3 — Load and preprocess
Set `IBM_DATA_PATH` to your actual `reduced_dataset.parquet` location.

In [16]:
!rm -rf /content/hybrid-gnn-lstm-fraud
!git clone -q https://github.com/Roya62/hybrid-gnn-lstm-fraud.git /content/hybrid-gnn-lstm-fraud
!ls /content/hybrid-gnn-lstm-fraud/lstm_solo/ibm/

config.py  lstm_only_model.py  utils.py


In [17]:
import os, sys

IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update

os.chdir('/content/hybrid-gnn-lstm-fraud/lstm_solo/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

import utils as ibm_utils
df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)
print(f'{len(df_ibm):,} transactions loaded.')

Loaded: 24,386,834 transactions
Downsampled: 29,757 fraud + 297,570 non-fraud
After preprocessing: 327,327 rows | Fraud rate: 0.0909
327,327 transactions loaded.


---
## Cell 4 — IBM: LSTM-only baseline
No `graph_strategy` loop — this model has no graph component, so it
runs once (not three times) using the same 5-fold splits as the hybrid
notebook's models. ~10–20 min on T4 GPU. Saves to Drive; re-running this
cell after a disconnect skips straight to the cached result.

In [18]:
import lstm_only_model as ibm_lstm_only
import pickle

save_path = '/content/drive/MyDrive/ibm_lstm_only_result.pkl'
if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        lstm_only_results_ibm = pickle.load(f)
    print("Skipping — already saved.")
else:
    lstm_only_results_ibm = ibm_lstm_only.run_all_strategies(df_ibm, cfg_ibm)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_only_results_ibm, f)
    print("Saved to Drive.")

print('\nIBM LSTM-only done.')

import pandas as pd
m = lstm_only_results_ibm['no_topology_lstm_only']['test_metrics']
print(f"TEST — F1 {m['f1']:.4f} | Prec {m['prec']:.4f} | Rec {m['rec']:.4f} | AUC {m['auc']:.4f} | AP {m['ap']:.4f}")


############################################################
# LSTM-only (no graph)
############################################################
  Fold 1/5
     TRAIN | F1 0.836 | P 0.860 | R 0.812 | AUC 0.983 | AP 0.908 | LL 0.1572 | Brier 0.0450
       VAL | F1 0.849 | P 0.878 | R 0.823 | AUC 0.982 | AP 0.915 | LL 0.1394 | Brier 0.0388
  Fold 2/5
     TRAIN | F1 0.838 | P 0.900 | R 0.783 | AUC 0.984 | AP 0.912 | LL 0.1585 | Brier 0.0454
       VAL | F1 0.839 | P 0.876 | R 0.804 | AUC 0.980 | AP 0.907 | LL 0.1822 | Brier 0.0531
  Fold 3/5
     TRAIN | F1 0.845 | P 0.902 | R 0.795 | AUC 0.985 | AP 0.918 | LL 0.1553 | Brier 0.0444
       VAL | F1 0.864 | P 0.897 | R 0.835 | AUC 0.986 | AP 0.929 | LL 0.1728 | Brier 0.0505
  Fold 4/5
     TRAIN | F1 0.842 | P 0.857 | R 0.828 | AUC 0.986 | AP 0.917 | LL 0.2053 | Brier 0.0604
       VAL | F1 0.838 | P 0.883 | R 0.798 | AUC 0.984 | AP 0.913 | LL 0.1783 | Brier 0.0524
  Fold 5/5
     TRAIN | F1 0.838 | P 0.907 | R 0.778 | AUC 0.985 | AP 0.91

---
# Part B — Sparkov

## Cell 5 — Load and preprocess
Set `SPARKOV_TRAIN_PATH` / `SPARKOV_TEST_PATH` to your actual
`fraudTrain.csv` / `fraudTest.csv` location. Independent of Part A —
safe to run this section on its own.

In [1]:
import sys
for _mod in ['config', 'utils', 'lstm_only_model']:
    sys.modules.pop(_mod, None)

In [2]:
import os, sys

SPARKOV_TRAIN_PATH = '/content/fraudTrain.csv'  # ← update if needed
SPARKOV_TEST_PATH  = '/content/fraudTest.csv'   # ← update if needed

os.chdir('/content/hybrid-gnn-lstm-fraud/lstm_solo/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg_sparkov = sparkov_cfg.CardFraudConfig()
cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

import utils as sparkov_utils
df_sparkov = sparkov_utils.load_and_preprocess(
    train_path=SPARKOV_TRAIN_PATH, test_path=SPARKOV_TEST_PATH, cfg=cfg_sparkov)
print(f'{len(df_sparkov):,} transactions loaded.')

Combined: 1,852,394 transactions
After downsample: 52,394 rows (fraud rate: 0.1842)
Final shape: (52394, 27)
52,394 transactions loaded.


---
## Cell 6 — Sparkov: LSTM-only baseline
Same ablation as Cell 4. Runs once (no `graph_strategy` loop).
~10–20 min on T4 GPU. Saves to Drive.

In [3]:
import lstm_only_model as sparkov_lstm_only
import pickle

save_path = '/content/drive/MyDrive/sparkov_lstm_only_result.pkl'
if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        lstm_only_results_sparkov = pickle.load(f)
    print("Skipping — already saved.")
else:
    lstm_only_results_sparkov = sparkov_lstm_only.run_all_strategies(df_sparkov, cfg_sparkov)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_only_results_sparkov, f)
    print("Saved to Drive.")

print('\nSparkov LSTM-only done.')

m = lstm_only_results_sparkov['no_topology_lstm_only']['test_metrics']
print(f"TEST — F1 {m['f1']:.4f} | Prec {m['prec']:.4f} | Rec {m['rec']:.4f} | AUC {m['auc']:.4f} | AP {m['ap']:.4f}")


############################################################
# LSTM-only (no graph)
############################################################
  Fold 1/5
     TRAIN | F1 0.994 | P 0.989 | R 0.999 | AUC 1.000 | AP 1.000 | LL 0.0215 | Brier 0.0065
       VAL | F1 0.961 | P 0.969 | R 0.954 | AUC 0.998 | AP 0.993 | LL 0.0544 | Brier 0.0146
  Fold 2/5
     TRAIN | F1 0.995 | P 0.992 | R 0.999 | AUC 1.000 | AP 1.000 | LL 0.0158 | Brier 0.0046
       VAL | F1 0.951 | P 0.955 | R 0.947 | AUC 0.997 | AP 0.990 | LL 0.0675 | Brier 0.0179
  Fold 3/5
     TRAIN | F1 0.996 | P 0.994 | R 0.998 | AUC 1.000 | AP 1.000 | LL 0.0124 | Brier 0.0035
       VAL | F1 0.959 | P 0.965 | R 0.952 | AUC 0.998 | AP 0.992 | LL 0.0560 | Brier 0.0140
  Fold 4/5
     TRAIN | F1 0.991 | P 0.983 | R 1.000 | AUC 1.000 | AP 1.000 | LL 0.0144 | Brier 0.0042
       VAL | F1 0.960 | P 0.959 | R 0.961 | AUC 0.999 | AP 0.994 | LL 0.0420 | Brier 0.0118
  Fold 5/5
     TRAIN | F1 0.996 | P 0.993 | R 0.998 | AUC 1.000 | AP 1.00

---
## Cell 7 — Combined summary (both datasets, if both were run)

In [4]:
import pandas as pd

rows = []
try:
    m = lstm_only_results_ibm['no_topology_lstm_only']['test_metrics']
    rows.append({'dataset': 'IBM', 'f1': round(m['f1'],4), 'prec': round(m['prec'],4),
                 'rec': round(m['rec'],4), 'auc': round(m['auc'],4), 'ap': round(m['ap'],4)})
except NameError:
    print('IBM not run in this session — skipping.')

try:
    m = lstm_only_results_sparkov['no_topology_lstm_only']['test_metrics']
    rows.append({'dataset': 'Sparkov', 'f1': round(m['f1'],4), 'prec': round(m['prec'],4),
                 'rec': round(m['rec'],4), 'auc': round(m['auc'],4), 'ap': round(m['ap'],4)})
except NameError:
    print('Sparkov not run in this session — skipping.')

df_summary = pd.DataFrame(rows)
os.makedirs('/content/outcomes', exist_ok=True)
df_summary.to_csv('/content/outcomes/lstm_only_summary.csv', index=False)
df_summary

IBM not run in this session — skipping.


,dataset,f1,prec,rec,auc,ap
0,Sparkov,0.9595,0.9831,0.9371,0.9989,0.9955


---
## Cell 8 — Download results

In [5]:
!cd /content/outcomes && zip -qr /content/outcomes_lstm_only.zip .
from google.colab import files
files.download('/content/outcomes_lstm_only.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>